## COCO Person Dataset Conversion to YOLO Format

In [5]:
import os
import json
import shutil
from tqdm import tqdm

base_dir = "../datasets/coco"
output_dir = "../datasets/coco-person"

target_classes = {1}
coco_id_to_yolo_id = {coco_id: i for i, coco_id in enumerate(sorted(target_classes))}
splits = ['train', 'val']

In [6]:
def coco_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    x_c = (x + w / 2) / img_w
    y_c = (y + h / 2) / img_h
    return [x_c, y_c, w / img_w, h / img_h]

In [ ]:
# Perform the conversion for each split
def process_split(split):
    print(f"Processing {split} split...")
    
    # Grabs and creates the necessary directories
    anno_path = os.path.join(base_dir, "annotations", f"instances_{split}2017.json")
    image_dir = os.path.join(base_dir, "images", f"{split}2017")
    out_img_dir = os.path.join(output_dir, "images", split)
    out_lbl_dir = os.path.join(output_dir, "labels", split)
    
    with open(anno_path) as f:
        coco = json.load(f)
    
    
    imgs = {img["id"]: img for img in coco["images"]}
    anns = [ann for ann in coco["annotations"] if ann["category_id"] in target_classes]
    
    
    labels_by_image = {}
    for ann in anns:
        img_id = ann["image_id"]
        img = imgs[img_id]
        file_name = img["file_name"]
        img_w, img_h = img["width"], img["height"]
        
        # Convert COCO bbox to YOLO format
        # YOLO format: [x_center, y_center, width, height]
        yolo_box = coco_to_yolo(ann["bbox"], img_w, img_h)
        
        #Map COCO category ID to YOLO class ID
        yolo_class = coco_id_to_yolo_id[ann["category_id"]]
        
        # Create the label line
        # YOLO format: class_id x_center y_center width height where all values are normalized to [0, 1]
        label_line = [yolo_class] + yolo_box
        
        if file_name not in labels_by_image:
            labels_by_image[file_name] = []
        labels_by_image[file_name].append(label_line)
    
    # For each image, create a corresponding label file
    for file_name, labels in tqdm(labels_by_image.items()):
        src_img = os.path.join(image_dir, file_name)
        dst_img = os.path.join(out_img_dir, file_name)
        dst_txt = os.path.join(out_lbl_dir, file_name.replace(".jpg", ".txt"))
        
        if not os.path.exists(src_img):
            continue
        
        # Copies files to the output directory
        shutil.copyfile(src_img, dst_img)
        
        # Writes the labels to a text file
        with open(dst_txt, "w") as f:
            for label in labels:
                f.write(" ".join([f"{x:.6f}" for x in label]) + "\n")

In [ ]:
for split in splits:
    process_split(split)

print("Done: COCO person subset converted to YOLO format.")

Processing train split...


100%|██████████| 64115/64115 [00:44<00:00, 1428.05it/s]


Processing val split...


100%|██████████| 2693/2693 [00:02<00:00, 907.62it/s] 


✅ Done: COCO threat subset converted to YOLO format.
